# **1) Download the necessary Libraries**

In [ ]:
!pip install -U transformers datasets peft trl accelerate bitsandbytes


# **2) Upload the Dataset files (cleaned_train and cleaned_valid)**

In [ ]:
from google.colab import files

# This will open a browser upload dialog window
uploaded = files.upload()

# Confirming successful upload
for filename in uploaded.keys():
    print(f'Successfully uploaded file "{filename}" with length {len(uploaded[filename])} bytes')

# **3) Loading the Models and trainings**

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig
from trl import SFTTrainer
import torch
import os

# =========================
# MODEL
# =========================

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# =========================
# DOWNLOADS FOLDER PATH
# =========================

downloads_folder = os.path.join(
    os.path.expanduser("~"),
    "Downloads",
    "qwen_model"
)

# Create folder if it doesn't exist
os.makedirs(downloads_folder, exist_ok=True)

print(f"Model will be downloaded to: {downloads_folder}")

# =========================
# LOAD DATASET
# =========================

dataset = load_dataset(
    "json",
    data_files={
        "train": "cleaned_train.jsonl",
        "validation": "cleaned_valid.jsonl"
    }
)

# =========================
# TOKENIZER
# =========================

print("Downloading/loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=downloads_folder,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token

# =========================
# 4-BIT CONFIG
# =========================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# =========================
# LOAD MODEL
# =========================

print("Downloading/loading model...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=downloads_folder,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded successfully!")

model.config.use_cache = False

# =========================
# LORA CONFIG
# =========================

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# =========================
# TRAINING ARGUMENTS
# =========================

training_args = TrainingArguments(
    output_dir="./socratic-model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    report_to="none"
)

# =========================
# TRAINER
# =========================

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
    args=training_args,
)

# =========================
# TRAIN
# =========================

print("Starting training...")

trainer.train()

# =========================
# SAVE TRAINED MODEL
# =========================

save_folder = os.path.join(
    os.path.expanduser("~"),
    "Downloads",
    "socratic-model"
)

os.makedirs(save_folder, exist_ok=True)

print("Saving trained model...")

trainer.model.save_pretrained(save_folder)
tokenizer.save_pretrained(save_folder)

print("Training complete!")
print(f"Downloaded model cache: {downloads_folder}")
print(f"Trained model saved at: {save_folder}")

# **4) Install the Package required for Ngrok hosting**

In [ ]:
# 1. Install required packages for hosting (With the torchao upgrade fix!)
!pip install flask-ngrok pyngrok flask-cors torchao --upgrade -q

# **5) The Netwroking Tunnel which creates a link used in the pycharm project to access and send data**

In [ ]:
# 1. Install required packages for hosting
!pip install flask-ngrok pyngrok flask-cors -q

from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

app = Flask(__name__)
CORS(app) # Allows external traffic from PyCharm

# 2. Authenticate ngrok (Get your free token from dashboard.ngrok.com)
# Sign up for a free account on ngrok and paste your token inside the quotes below!
NGROK_TOKEN = "3EEjSx55H0waGw8iZVJjSBXsQwM_36vVN8nXgyNwu9gkgFRfV"
ngrok.set_auth_token(NGROK_TOKEN)

# 3. Load the model cleanly using Colab's fast VRAM
base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(base_model_name, torch_dtype=torch.float16, device_map="auto")

# Point this directly to your final checkpoint directory in Colab!
model = PeftModel.from_pretrained(base_model, "./socratic-model/checkpoint-171")
model.eval()

@app.route("/generate", methods=["POST"])
def generate():
    data = request.json
    user_input = data.get("prompt", "")

    # NEW GROUNDED SYSTEM PROMPT: Gives the model strict guardrails
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful, logical Socratic computer science tutor. "
                "Your goal is to guide students to the answer by asking informative hints. "
                "Do not repeat the same question twice. If the user asks something completely "
                "unrelated to programming, answer it normally and politely without being Socratic."
            )
        },
        {"role": "user", "content": user_input}
    ]

    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
    input_length = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.6,          # Lowered slightly (0.7 -> 0.6) to make it more focused/less chaotic
            top_p=0.85,               # Keeps it tracking high-probability, logical words
            repetition_penalty=1.2,   # CRITICAL: Punishes the model mathematically if it tries to repeat or loop!
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][input_length:]
    response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    return jsonify({"response": response_text})

# 4. Open the Tunnel and Run
public_url = ngrok.connect(5000)
print(f"\n🚀 COPY THIS URL AND PASTE IT INTO PYCHARM:\n{public_url}\n")

app.run()